## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re
import math
import statistics


C:\Users\skyei\AppData\Roaming\Python\Python312\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

# sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [16]:
# This is Cell #5

#TODO: Create a list of unique characters from the text sequence
vocab = list(set(sequence))

#TODO: Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
count = 0
char_to_idx = {}
idx_to_char = {}
for char in vocab:
    char_to_idx[char] = count
    idx_to_char[count] = char
    count += 1

#TODO: Convert the entire text based data into numerical data
def to_numerical(string):
    ans = []
    for char in string:
        ans.append(char_to_idx[char])
    return ans

data = to_numerical(sequence)


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [17]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 100  # Length of each input sequence
stride = 10            # Stride for creating sequences (give)
embedding_dim = 32     # Dimension of character embeddings
hidden_size = 256      # Number of features in the hidden state of the RNN
learning_rate = 0.001  # Learning rate for the optimizer
num_epochs = 3         # Number of epochs to train
batch_size = 64
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below
> 
> Sequence length -- Determines how long each sequence used within the dataset is \
> stride -- Determines the "gap" in characters between the start of each sequence within the complete input \
> embedding_dim -- determines how many dimensions are created when embedding the categorical data of the input into continuous in the RNN \
> hidden_size -- Determines how many features are saved in the hidden state \
> Learning rate -- Determines the learning rate for the optimizer that we will set up below \
> num_epochs -- will determine how many epochs we train for \
> batch_size -- determines how large each batch used in training will be \
> vocab size -- how many characters are in the vocabulary? 


## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [18]:
# This is Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

#TODO: Convert the data into a pytorch tensor and split the data into 90:10 ratio

#train_test_shape = [0.9,0.1]
#train_data, test_data = torch.utils.data.random_split(data_tensor, train_test_shape)

train_data = data_tensor[:math.ceil(0.9*len(data_tensor))]
test_data = data_tensor[math.ceil(0.9*len(data_tensor)):]

print(len(train_data))
print(len(test_data))


2802826
311425


In [35]:
print(type(train_data))

<class 'torch.Tensor'>


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [19]:
# This is Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

#TODO: Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle = True, drop_last = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle = False, drop_last=True)

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [12]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [20]:
# This is Cell #12

#TODO: Initialize your RNN model
model = CharRNN(input_size, hidden_size, output_size, embedding_dim)

#TODO: Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

#TODO: Initialize your optimizer passing your model parameters and training hyperparameters
parameters = []
"""for i in model.parameters():
    parameters.append(i)
#parameters.append(num_epochs)
#parameters.append(batch_size)"""
optimizer = optim.Adam(model.parameters(), lr = learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [21]:
# This is Cell #13
print(num_epochs)
for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)
    #print(tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"))
    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

3


Epoch 1/3:   0%|          | 0/4379 [00:00<?, ?it/s]C:\Users\skyei\AppData\Local\Temp\ipykernel_52088\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\skyei\AppData\Local\Temp\ipykernel_52088\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/3: 100%|██████████| 4379/4379 [03:53<00:00, 18.77it/s]


Epoch [1/3], Loss: 1.6216, Accuracy: 52.51%


Epoch 2/3: 100%|██████████| 4379/4379 [03:46<00:00, 19.35it/s]


Epoch [2/3], Loss: 1.4236, Accuracy: 57.99%


Epoch 3/3: 100%|██████████| 4379/4379 [03:59<00:00, 18.31it/s]

Epoch [3/3], Loss: 1.3946, Accuracy: 58.80%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [15]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [14]:
# This is Cell #15

with torch.no_grad():
    #TODO: Write the testing loop for your trained model by refering to the training loop code given to you above
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)
    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(test_loader), total=total_batches):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        #hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        #optimizer.zero_grad()

        #loss.backward()

        #optimizer.step()

        
        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
        

    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

  0%|          | 0/4379 [00:00<?, ?it/s]C:\Users\skyei\AppData\Local\Temp\ipykernel_79096\1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
C:\Users\skyei\AppData\Local\Temp\ipykernel_79096\1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
 11%|█         | 486/4379 [00:07<00:58, 66.47it/s]

Test Loss: 0.1631, Test Accuracy: 56.82%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [ ]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    #print (probabilities)
    #print(sampled_idx)
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    if k<=0:
        return start_text

    start_idx = []
    for char in start_text:
        start_idx.append(char_to_idx[char])
    #char_to_idx(start_text)

    hidden = None

    #start_input = torch.tensor(start_idx[-n:], dtype = torch.long).unsqueeze(0).to(device)

    
    #print(start_input.shape)

    for i in range(k):

        start_input = torch.tensor(start_idx[-n:], dtype = torch.long).unsqueeze(0).to(device)
        logits, hidden = model(start_input, hidden)
        #print(logits.tolist()[-1])
        #_, predicted_indices = torch.max(logits, dim=2)

    #print(predicted_indices)
    #print(torch.squeeze(predicted_indices,1))
    #print(sample_from_output(torch.squeeze(predicted_indices,1), temperature))

        sample_idx = sample_from_output(logits[-1][-1:], temperature)
        #print(sample_idx)
        next_char = idx_to_char[int(sample_idx[-1])]

        start_text = start_text + next_char
        start_idx.append(sample_idx)

    return start_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
[[0.2301875650882721, -0.6697705388069153, 0.48252856731414795, 1.9384874105453491, -4.536994457244873, -0.5858220458030701, 5.2986578941345215, -2.1097280979156494, 1.286301612854004, 1.5922677516937256, -1.0906693935394287, -0.23418134450912476, -3.593484878540039, -3.582435131072998, 5.948814868927002, 2.1607296466827393, -3.553722620010376, -5.364564418792725, -1.0092494487762451, -2.695516586303711, -4.2353363037109375, 3.5408835411071777, -4.035055160522461, 4.744868278503418, 0.9233510494232178, -6.5703630447387695, -9.365571975708008, -1.8580710887908936, -4.34369421005249, 3.1117522716522217, -0.7484751343727112, -1.0292035341262817, -5.280960559844971, 5.926589012145996, 0.34986862540245056, 0.5437015891075134, -1.134070634841919], [-0.9605695605278015, -4.134075164794922, -0.7162778377532959, 1.0816272497177124, -2.1565639972686768, -4.544921398162842, 5.489431858062744, -0.26963549852371216, -0.23581665754318237, 2.2801206111907

ValueError: invalid literal for int() with base 10: ''

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


When training with the sequence `"abcdefghijklmnopqrstuvwxyz" * 100`, the training was much quicker and I got much higher accuracy than with the longer, more complex warandpeace.txt. 
My final loss value for abcdefghij...[etc.] was roughly 1.5  (This was before changing the hyperparameters specifically for war and peace. I didn't save the actual value, and retraining with warandpeace.txt will take me another hour). My final loss value for warandpeace.txt was 1.3719 in training.

I didn't see much difference bettween the two texts or the temperature parameter, as there doesn't seem to be much of a pattern as to what the model is predicting as text.